In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_log_error
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

plt.style.use('seaborn-v0_8-pastel')
sns.set_palette("pink")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.facecolor'] = '#fff7f9'

In [2]:
df_train = pd.read_csv('train.csv', parse_dates=['date'])
df_test = pd.read_csv('test.csv', parse_dates=['date'])
df_holidays = pd.read_csv('holidays_events.csv', parse_dates=['date'])
df_oil = pd.read_csv('oil.csv', parse_dates=['date'])
df_stores = pd.read_csv('stores.csv')
df_trans = pd.read_csv('transactions.csv', parse_dates=['date'])
df_subm = pd.read_csv('sample_submission.csv')

print(f"Train: {df_train.shape}, Test: {df_test.shape}")

Train: (3000888, 6), Test: (28512, 5)


In [3]:
# stores
df_train = df_train.merge(df_stores, on='store_nbr', how='left')
df_test = df_test.merge(df_stores, on='store_nbr', how='left')

# oil
df_oil = df_oil.set_index('date')['dcoilwtico'].fillna(method='ffill').fillna(method='bfill')
df_train = df_train.merge(df_oil, on='date', how='left')
df_test = df_test.merge(df_oil, on='date', how='left')
df_train['dcoilwtico'] = df_train['dcoilwtico'].fillna(df_train['dcoilwtico'].median())
df_test['dcoilwtico'] = df_test['dcoilwtico'].fillna(df_train['dcoilwtico'].median())

# transactions
df_trans_indexed = df_trans.set_index(['date', 'store_nbr'])['transactions']
df_train = df_train.merge(df_trans_indexed, on=['date', 'store_nbr'], how='left')
df_test = df_test.merge(df_trans_indexed, on=['date', 'store_nbr'], how='left')
df_train['transactions'] = df_train['transactions'].fillna(0)
df_test['transactions'] = df_test['transactions'].fillna(0)

# holidays
holidays = df_holidays.copy()
holidays['is_holiday'] = 1
national = holidays[holidays['locale'] == 'National'][['date', 'is_holiday']]
regional = holidays[holidays['locale'] == 'Regional'][['date', 'locale_name', 'is_holiday']].rename(columns={'locale_name': 'state'})
local = holidays[holidays['locale'] == 'Local'][['date', 'locale_name', 'is_holiday']].rename(columns={'locale_name': 'city'})

for df in [df_train, df_test]:
    df = df.merge(national, on='date', how='left')
    df = df.merge(regional, on=['date', 'state'], how='left', suffixes=('', '_reg'))
    df = df.merge(local, on=['date', 'city'], how='left', suffixes=('', '_loc'))
    df['is_holiday'] = df[['is_holiday', 'is_holiday_reg', 'is_holiday_loc']].max(axis=1).fillna(0).astype(int)
    df.drop(columns=['is_holiday_reg', 'is_holiday_loc'], inplace=True, errors='ignore')

In [4]:
for df in [df_train, df_test]:
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)

# Earthquake
earthquake = pd.to_datetime('2016-04-16')
for df in [df_train, df_test]:
    df['days_after_earthquake'] = (df['date'] - earthquake).dt.days.clip(0, 14)

# School season
df_train['school_season'] = 0
df_test['school_season'] = 0
for year in range(2013, 2018):
    start = f'{year}-08-15'
    end = f'{year}-09-15'
    mask_train = (df_train['date'] >= start) & (df_train['date'] < end)
    df_train.loc[mask_train, 'school_season'] = 1
    if year == 2017:
        mask_test = (df_test['date'] >= start) & (df_test['date'] < end)
        df_test.loc[mask_test, 'school_season'] = 1

# Payday
for df in [df_train, df_test]:
    df['payday'] = ((df['date'].dt.day == 15) | (df['date'].dt.is_month_end)).astype(int)

# Fourier
for df in [df_train, df_test]:
    df['dayofyear'] = df['date'].dt.dayofyear
    df['sin_week'] = np.sin(2 * np.pi * df['dayofyear'] / 7)
    df['cos_week'] = np.cos(2 * np.pi * df['dayofyear'] / 7)
    df['sin_year'] = np.sin(2 * np.pi * df['dayofyear'] / 365)
    df['cos_year'] = np.cos(2 * np.pi * df['dayofyear'] / 365)

In [5]:
def add_lag_rolling_features(df):
    df = df.sort_values(['store_nbr', 'family', 'date'])
    group_cols = ['store_nbr', 'family']
    
    # Лаги
    for lag in [1, 3, 7, 14, 28, 365]:
        df[f'lag_{lag}'] = df.groupby(group_cols)['sales'].shift(lag)
    
    # Rolling
    for w in [7, 14, 28]:
        df[f'roll_mean_{w}'] = df.groupby(group_cols)['sales'].shift(1).rolling(window=w, min_periods=1).mean()
        df[f'roll_std_{w}']  = df.groupby(group_cols)['sales'].shift(1).rolling(window=w, min_periods=1).std()
    
    return df

# Сначала train
df_train = add_lag_rolling_features(df_train)

# Потом конкатенируем с test и применяем снова
df_full = pd.concat([df_train, df_test], ignore_index=True)
df_full = df_full.sort_values(['store_nbr', 'family', 'date'])
df_full = add_lag_rolling_features(df_full)

# Разделяем обратно
df_train = df_full[df_full['date'] < '2017-08-16'].copy()
df_test  = df_full[df_full['date'] >= '2017-08-16'].copy()

# Заполняем NaN
lag_roll_cols = [c for c in df_train.columns if ('lag_' in c) or ('roll_' in c)]
df_train[lag_roll_cols] = df_train[lag_roll_cols].fillna(0)
df_test[lag_roll_cols]  = df_test[lag_roll_cols].fillna(0)

In [6]:
# ЯЧЕЙКА 6 — Кодирование категориальных признаков (БЕЗ category_encoders)
from sklearn.preprocessing import LabelEncoder

# Список категориальных колонок
cat_cols = ['city', 'state', 'type', 'family', 'store_nbr']

# Словарь для сохранения энкодеров
encoders = {}

# Применяем LabelEncoder к каждой колонке
for col in cat_cols:
    le = LabelEncoder()
    # Обучаем на train + test, чтобы не было новых значений
    combined = pd.concat([df_train[col], df_test[col]], axis=0).astype(str)
    le.fit(combined)
    
    df_train[col + '_enc'] = le.transform(df_train[col].astype(str))
    df_test[col + '_enc'] = le.transform(df_test[col].astype(str))
    
    encoders[col] = le

# Удаляем старые строковые колонки
df_train = df_train.drop(columns=cat_cols)
df_test = df_test.drop(columns=cat_cols)

# Логарифм продаж
df_train['sales_log'] = np.log1p(df_train['sales'])

print("Категории закодированы LabelEncoder'ом!")
print(f"Новые колонки: {[c for c in df_train.columns if '_enc' in c]}")
df_train.head()

Категории закодированы LabelEncoder'ом!
Новые колонки: ['city_enc', 'state_enc', 'type_enc', 'family_enc', 'store_nbr_enc']


,id,date,sales,onpromotion,cluster,dcoilwtico,transactions,year,month,day,...,roll_mean_14,roll_std_14,roll_mean_28,roll_std_28,city_enc,state_enc,type_enc,family_enc,store_nbr_enc,sales_log
0,0,2013-01-01,0.0,0,13,93.14,0.0,2013,1,1,...,0.000000,0.000000,0.000000,0.000000,18,12,3,0,0,0.000000
1,1782,2013-01-02,2.0,0,13,93.14,2111.0,2013,1,2,...,0.000000,0.000000,0.000000,0.000000,18,12,3,0,0,1.098612
2,3564,2013-01-03,3.0,0,13,92.97,1833.0,2013,1,3,...,1.000000,1.414214,1.000000,1.414214,18,12,3,0,0,1.386294
3,5346,2013-01-04,3.0,0,13,93.12,1863.0,2013,1,4,...,1.666667,1.527525,1.666667,1.527525,18,12,3,0,0,1.386294
4,7128,2013-01-05,5.0,0,13,53.38,1509.0,2013,1,5,...,2.000000,1.414214,2.000000,1.414214,18,12,3,0,0,1.791759


In [7]:
# ЯЧЕЙКА 7 — Обновляем список фичей
features = [col for col in df_train.columns if col not in ['id', 'date', 'sales', 'sales_log']]
print(f"Используется {len(features)} признаков")

Используется 35 признаков


In [8]:
models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'DecisionTree': DecisionTreeRegressor(max_depth=10, random_state=42),
    'HistGradientBoosting': HistGradientBoostingRegressor(
        max_iter=300, max_depth=10, learning_rate=0.05, random_state=42
    )
}

features = df_train.columns.drop(['id', 'date', 'sales', 'sales_log'])

tscv = TimeSeriesSplit(n_splits=3)
results = {}

for name, model in models.items():
    print(f"\n=== {name} ===")
    scores = []
    for fold, (train_idx, val_idx) in enumerate(tscv.split(df_train)):
        train_set = df_train.iloc[train_idx]
        val_set   = df_train.iloc[val_idx]
        
        X_train, y_train = train_set[features], train_set['sales_log']
        X_val,   y_val   = val_set[features],   val_set['sales_log']
        
        model.fit(X_train, y_train)
        pred_log = model.predict(X_val).clip(0)
        pred = np.expm1(pred_log)
        true = np.expm1(y_val)
        score = np.sqrt(mean_squared_log_error(true, pred))
        scores.append(score)
        print(f"Fold {fold+1}: RMSLE = {score:.4f}")
    
    avg_score = np.mean(scores)
    results[name] = avg_score
    print(f"{name} → Average RMSLE: {avg_score:.4f}")

# Итог
print("\n=== FINAL RESULTS ===")
for name, score in results.items():
    print(f"{name}: {score:.4f}")


=== LinearRegression ===
Fold 1: RMSLE = 2.0346
Fold 2: RMSLE = 2.0250
Fold 3: RMSLE = 2.4834
LinearRegression → Average RMSLE: 2.1810

=== Ridge ===
Fold 1: RMSLE = 2.0346
Fold 2: RMSLE = 2.0250
Fold 3: RMSLE = 2.4834
Ridge → Average RMSLE: 2.1810

=== DecisionTree ===
Fold 1: RMSLE = 0.4551
Fold 2: RMSLE = 0.4426
Fold 3: RMSLE = 0.4372
DecisionTree → Average RMSLE: 0.4450

=== HistGradientBoosting ===
Fold 1: RMSLE = 0.4266
Fold 2: RMSLE = 0.4130
Fold 3: RMSLE = 0.4009
HistGradientBoosting → Average RMSLE: 0.4135

=== FINAL RESULTS ===
LinearRegression: 2.1810
Ridge: 2.1810
DecisionTree: 0.4450
HistGradientBoosting: 0.4135
